# 07 — Frozen held-out test

Training ve validation tamamlandı. Bu notebook yalnızca ayrı `held-out-test-v1` akışını yönetir.
Tüm çalıştırma anahtarları kapalıdır. Kaynak/input ZIP hash'leri sabittir; mevcut modeller ve validation probe girdileri yeniden kullanılır.

1. Paketleri hash doğrulamasıyla aç.
2. Gerçek T4 readiness: input/target kontrolü, dört checkpoint, validation-only teknik inference; **test skoru yok**.
3. Readiness sonrasında açıkça başlatılan test: **122 koşul**, aynı tahminlerden iki cohort / 244 skor.
4. İş bitince kayıtlı tahminlerden audit/rapor; yeni inference yok.

[Protokol](../docs/held-out-test-protocol.md) · [Çalıştırma kaydı](../docs/held-out-test-execution.md)

Mevcut eşleşen T4 runtime kullanılır. Oturum kapanmışsa aşağıdaki ayrı ve kapalı cold-restoration adımı kayıtlı paket sürümlerini ve arşivleri geri yükler. Python/Torch/CUDA/GPU farkında işlem durur; hiçbir karşılaştırma kontrolü gevşetilmez. Yeni training veya Drive temizliği içermez.

In [ ]:
from pathlib import Path
import os
import sys

MOUNT_DRIVE = False
RUN_RESTORE = False
RUN_RUNTIME_RESTORE = False
RUN_READINESS = False
RUN_TEST = False
RUN_ANALYSIS = False

ROOT = Path('/content/cmpe586-turkish-punctuation')
STORAGE = Path('/content/drive/MyDrive/cmpe586-turkish-punctuation')
TRANSFER = STORAGE / 'preparation/held-out-test-v1/transfer-v1'
INPUT_ROOT = Path('/content/cmpe586-held-out-inputs-v1')
VALIDATION_ROOT = Path('/content/cmpe586-evaluation-inputs-noise-v1')
READINESS = STORAGE / 'preparation/held-out-test-v1/runtime-v1'
RUN = STORAGE / 'artifacts/evaluations/held-out-test-v1'
ANALYSIS = STORAGE / 'artifacts/evaluations/held-out-test-v1/analysis-v1'

BUNDLES = {'inputs': {'filename': 'held-out-inputs-v1.zip', 'sha256': '18953b8eb131c18d261aee909ba7d1092f4178d2384b02d2139eedb4038159f1'}, 'source': {'filename': 'held-out-source-v1.zip', 'sha256': '115877201eb35cb81f3e24e870a9a0acabc44e42933b399518db56ed10ebb9b2'}}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

os.environ['PYTHONPATH'] = str(ROOT / 'src')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Tüm execution anahtarları varsayılan olarak kapalı. Test yalnızca RUN_TEST=True ile başlar.')

## Expired runtime restoration — ayrı, varsayılan kapalı

T4 / runtime version `2026.07` seçilir. Önceki oturum kapanmışsa Drive mount ve ilk path hücresi çalıştıktan sonra bu adım kullanılabilir. `restore_heldout_colab.py` ile eski `package_evaluation_inputs.py` dosyaları transfer klasöründe SHA ile doğrulanır. Mevcut validation ZIP'i ve seçilmiş modeller yeniden kullanılır. Published checkpoint yalnızca runtime cache'te yoksa sabit revision'dan alınır. Paketler kayıtlı constraints'e geri getirilir; noise üretilmez, forward pass veya skor hesaplanmaz. Bu adım readiness değildir.

Notebook'taki Git kopyasının bütün execution anahtarları kapalı tutulur. Yeni bir restoration receipt yolu seçilir; önceki kanıt üzerine yazılmaz.

In [ ]:
if RUN_RUNTIME_RESTORE:
    import hashlib
    import signal
    import subprocess
    from datetime import datetime, timezone
    helper = TRANSFER/'restore_heldout_colab-r2.py'
    if hashlib.sha256(helper.read_bytes()).hexdigest() != '063f3939f2b91124d3b565c531978dbe82e54e58fc84f36097d4f9c05af75554':
        raise ValueError('Runtime restoration helper SHA differs.')
    RESTORE_OUTPUT = STORAGE/'preparation/held-out-test-v1'/('restore-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    RESTORE_OUTPUT.mkdir(parents=True)
    command = [sys.executable,'-u',str(helper),'--project',str(ROOT),'--storage',str(STORAGE),
        '--inputs',str(INPUT_ROOT),'--validation',str(VALIDATION_ROOT),
        '--validation-packager-sha256','59897a4a17ae9a3e29aceb7d07ce0590ec949dfd50b42b087b5ca20eeab2c4bd',
        '--receipt',str(RESTORE_OUTPUT/'restoration.json')]
    with subprocess.Popen(command,cwd='/content',env=os.environ.copy(),stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True) as process:
        try:
            with (RESTORE_OUTPUT/'restoration.log').open('a') as log:
                for line in process.stdout:
                    print(line,end='',flush=True)
                    log.write(line)
                    log.flush()
            code=process.wait()
        except KeyboardInterrupt:
            os.killpg(process.pid,signal.SIGINT)
            process.wait()
            raise
        if code:
            raise subprocess.CalledProcessError(code,command)
else:
    print('Cold runtime restoration kapalı.')

In [ ]:
import hashlib
import signal
import subprocess
import tempfile
import zipfile


def sha256(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def run(command):
    process = subprocess.Popen([sys.executable, '-u', *map(str, command)],
        cwd=ROOT, env=os.environ.copy(), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
        if code:
            raise RuntimeError(f'İşlem durdu: exit={code}; yukarıdaki hatayı incele.')
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGINT)
        process.wait()
        raise


if RUN_RESTORE:
    # Only bootstrap the restorer after verifying the complete source archive.
    source = TRANSFER / BUNDLES['source']['filename']
    if sha256(source) != BUNDLES['source']['sha256']:
        raise ValueError('Source ZIP hash farklı; çalıştırma durduruldu.')
    ROOT.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='held-out-bootstrap-') as temporary:
        bootstrap = Path(temporary) / 'package_heldout_execution.py'
        with zipfile.ZipFile(source) as archive:
            bootstrap.write_bytes(archive.read('scripts/package_heldout_execution.py'))
        run([bootstrap, 'restore', '--archive', source, '--sha256', BUNDLES['source']['sha256'],
             '--destination', ROOT])
    run([ROOT/'scripts/package_heldout_execution.py', 'restore',
         '--archive', TRANSFER/BUNDLES['inputs']['filename'], '--sha256', BUNDLES['inputs']['sha256'],
         '--destination', INPUT_ROOT])
else:
    print('Restore kapalı; dosyalara dokunulmadı.')

## Readiness — test sonucu üretmez

Bütün test görünümlerini hedef/byte/tokenization açısından doğrular. Sabit validation batch'leri ve validation kelimelerinden maksimum planlanan boyuta padding uygulanmış bir batch ile dört modelde teknik GPU kontrolü yapar. Test girdilerini modele vermez, skor veya optimizer update üretmez.

Mevcut validation bundle/cache ve seçilmiş Drive modelleri gereklidir. Tamamlanmış aynı readiness varsa probe tekrarlanmaz. Hash/runtime farkları atlanmaz.

In [ ]:
if RUN_READINESS:
    run([ROOT/'scripts/ready_heldout_evaluation.py', '--input-root', INPUT_ROOT,
         '--validation-input-root', VALIDATION_ROOT, '--storage-root', STORAGE, '--output', READINESS])
else:
    print('Readiness çalıştırılmadı.')

## Held-out inference — ayrı ve açık başlatma

Her model için yeni clean test anchor. Ana üç modelde 39'ar koşul; ablation'da clean + dört medium/seed586 koşulu. Aynı clean-length sırası/batch üyeliği bütün modellerde korunur. Sensitivity aynı tahminleri filtreler.

Kesinti sonrasında yalnızca hash ve raw-target kontrolü geçen tamamlanmış koşullar tekrar kullanılabilir. Değişmiş bir runtime, kaynak, model veya input ile resume yapılmaz. Bu hücre tamamlanınca `RUN_ANALYSIS` ile sonuçlar yeni inference olmadan doğrulanır.

In [ ]:
if RUN_TEST:
    run([ROOT/'scripts/evaluate_heldout.py', '--input-root', INPUT_ROOT,
         '--storage-root', STORAGE, '--readiness', READINESS, '--allow-test-scoring'])
else:
    print('Test scoring kapalı; inference başlatılmadı.')

In [ ]:
if RUN_ANALYSIS:
    run([ROOT/'scripts/analyze_heldout.py', '--input-root', INPUT_ROOT,
         '--run-directory', RUN, '--output', ANALYSIS])
else:
    print('Sonuç audit/raporu çalıştırılmadı.')

In [ ]:
import json
for label, path in [('Readiness', READINESS/'readiness.json'), ('Inference', RUN/'progress.json')]:
    if path.exists():
        item = json.loads(path.read_text())
        print(label, {key: item[key] for key in ('status', 'identity', 'completed_conditions', 'planned_conditions', 'last_committed') if key in item})
    else:
        print(label, 'kalıcı kayıt yok; çalıştırıldığı varsayılmıyor.')
print('Bu hücre salt okunur; deney başlatmaz.')